In [1]:
"""
Hulu-Med-4B analysis script for Cholec80 dataset.
Generates answers for surgical questions using Hulu-Med model.
"""

import torch
from transformers import AutoModelForCausalLM, AutoProcessor
import json
from tqdm import tqdm

# Load model and processor
MODEL_PATH = "ZJU-AI4H/Hulu-Med-4B"
dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32
device_map = "auto" if torch.cuda.is_available() else None

print("Loading Hulu-Med-4B model...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    trust_remote_code=True,
    torch_dtype=dtype,
    device_map=device_map,
    attn_implementation="sdpa",   # safer than flash_attention_2 unless flash-attn is installed
)

processor = AutoProcessor.from_pretrained(
    MODEL_PATH,
    trust_remote_code=True,
)

tokenizer = processor.tokenizer
model.eval()

print("Model loaded successfully!")

/home/as5606/miniconda3/envs/hulu-med-clean/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading Hulu-Med-4B model...
[2026-06-15 13:43:54,396] [INFO] [real_accelerator.py:219:get_accelerator] Setting ds_accelerator to cuda (auto detect)


/home/as5606/miniconda3/envs/hulu-med-clean/bin/../lib/gcc/x86_64-conda-linux-gnu/14.3.0/../../../../x86_64-conda-linux-gnu/bin/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status
/home/as5606/miniconda3/envs/hulu-med-clean/bin/../lib/gcc/x86_64-conda-linux-gnu/14.3.0/../../../../x86_64-conda-linux-gnu/bin/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status


config.vision_encoder None
Build from config


Loading checkpoint shards: 100%|███████████████████████████████████████████████████████████████████████████████| 2/2 [00:19<00:00,  9.87s/it]


Model loaded successfully!


In [2]:

def hulumed_generate(conversation, modal="text", max_new_tokens=1024, temperature=0.0):
    inputs = processor(
        conversation=conversation,
        add_system_prompt=True,
        add_generation_prompt=True,
        return_tensors="pt",
    )

    if "modals" not in inputs:
        inputs["modals"] = [modal]

    model_device = next(model.parameters()).device
    use_cuda = model_device.type == "cuda"

    fixed_inputs = {}
    for k, v in inputs.items():
        if isinstance(v, torch.Tensor):
            if k == "pixel_values":
                dtype = torch.bfloat16 if use_cuda else torch.float32
                fixed_inputs[k] = v.to(device=model_device, dtype=dtype)
            else:
                fixed_inputs[k] = v.to(model_device)
        else:
            fixed_inputs[k] = v

    generation_kwargs = {
        "do_sample": temperature > 0,
        "max_new_tokens": max_new_tokens,
        "use_cache": True,
        "pad_token_id": tokenizer.eos_token_id,
    }

    if temperature > 0:
        generation_kwargs["temperature"] = temperature

    with torch.inference_mode():
        output_ids = model.generate(
            **fixed_inputs,
            **generation_kwargs,
        )

    return processor.batch_decode(
        output_ids,
        skip_special_tokens=True,
        use_think=False,
    )[0].strip()


In [6]:
# Load test dataset
print("Loading test dataset...")
with open('/home/as5606/Datasets/Cholec_text_grounded_questions/cholec80_with_grounded_questions.json', 'r') as f:
    test_data = json.load(f)

print(f"Loaded {len(test_data)} test samples")

# Store results


Loading test dataset...
Loaded 1402 test samples


In [8]:
test_data[0]['grounded_qa_pairs']

[{'question': 'What colour is the scissors?', 'answer': 'silver/grey'},
 {'question': 'Is the scissors on the left or right side of the image?',
  'answer': 'right'},
 {'question': 'Which direction is the tip of the scissors pointing?',
  'answer': 'left'},
 {'question': 'How many surgical instruments are visible in the image?',
  'answer': '1'}]

In [5]:
results = []

print("Generating answers for all questions...")
for item in tqdm(test_data, desc="Processing"):
    # Extract data
    item_id = item['id']
    frame_path = item['image']
    original_question = item['conversations'][0]['value'].replace('\n<image>', '').strip()

    # Create conversation for Hulu-Med
    conversation = [
        {
            "role": "user",
            "content": [
                {
                    "type": "image",
                    "image": {
                        "image_path": frame_path,
                    },
                },
                {
                    "type": "text",
                    "text": original_question,
                },
            ],
        }
    ]

    try:
        # Generate answer
        answer = hulumed_generate(conversation, modal="image", max_new_tokens=256)

        # Store result
        result = {
            "id": item_id,
            "image": frame_path,
            "question": original_question,
            "answer": answer
        }
        print(result['answer'])
        results.append(result)
        break

    except Exception as e:
        print(f"Error processing {item_id}: {e}")
        continue

Generating answers for all questions...


Processing:   0%|                                                                                     | 0/7652 [00:00<?, ?it/s]/home/as5606/miniconda3/envs/hulu-med-clean/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:631: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/home/as5606/miniconda3/envs/hulu-med-clean/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:636: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/home/as5606/miniconda3/envs/hulu-med-clean/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:653: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` --

no


In [ ]:
results = []

print("Generating Hulu-Med answers for grounded questions...")
for item in tqdm(test_data, desc="Processing"):
    # Extract data
    item_id = item['id']
    frame_path = item['image']
    original_question = item['conversations'][0]['value'].replace('\n<image>', '').strip()
    original_answer = item['conversations'][1]['value']
    
    # Extract the grounded QA pairs
    grounded_qa_pairs = item.get('grounded_qa_pairs', [])
    

    
    try:        
        # Generate Hulu-Med answers for each grounded question
        grounded_with_hulu = []
        for grounded_item in grounded_qa_pairs:
            grounded_question = grounded_item['question']
            original_grounded_answer = grounded_item['answer']
            
            # Create conversation for grounded question
            grounded_conversation = [
                {
                    "role": "user",
                    "content": [
                        {
                            "type": "image",
                            "image": {
                                "image_path": frame_path,
                            },
                        },
                        {
                            "type": "text",
                            "text": grounded_question,
                        },
                    ],
                }
            ]
            
            # Generate Hulu-Med answer for grounded question
            hulu_grounded_answer = hulumed_generate(grounded_conversation, modal="image", max_new_tokens=256)
            
            grounded_with_hulu.append({
                "question": grounded_question,
                "original_answer": original_grounded_answer,
                "hulu_med_answer": hulu_grounded_answer
            })
        
        # Store result
        result = {
            "id": item_id,
            "image": frame_path,
            "original_question": original_question,
            "original_answer": original_answer,
            "grounded_qa_with_hulu_answers": grounded_with_hulu
        }
        results.append(result)
        
    except Exception as e:
        print(f"Error processing {item_id}: {e}")
        continue


Generating Hulu-Med answers for grounded questions...


Processing:   0%|                                                                                     | 0/1402 [00:00<?, ?it/s]/home/as5606/miniconda3/envs/hulu-med-clean/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:631: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/home/as5606/miniconda3/envs/hulu-med-clean/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:636: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/home/as5606/miniconda3/envs/hulu-med-clean/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:653: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` --

In [ ]:
# Save to JSON
output_path = '/home/as5606/Datasets/Cholec_text_grounded_questions/hulu_med_4B_grounded_answers.json'
with open(output_path, 'w') as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print(f"\n✓ Saved {len(results)} results to {output_path}")
print(f"Sample result: {results[0] if results else 'No results'}")
